In [ ]:
from pathlib import Path
import sys
import os

current_directory = Path(os.getcwd())
print(current_directory)
root_directory = current_directory.parent.parent
print(root_directory)

sys.path.append(str(root_directory))

In [ ]:
from pathlib import Path
import typing as t
import re
from pprint import pprint
import coq_serapy as c
import json
import pickle

from src.config import CONFIG
from src.coq_serapy_util import LemmaLocation, Coq
from src.dataset import Example, COQGYM_TEST_SAMPLED_DATASET, COQGYM_TEST_SAMPLED_PERFECT_PREFIX

In [ ]:
for example in COQGYM_TEST_SAMPLED_DATASET:
    print(example.name)
    print(COQGYM_TEST_SAMPLED_PERFECT_PREFIX.get(example.name, None))
    print()

In [ ]:
TEST_EXAMPLE = next(example for example in COQGYM_TEST_SAMPLED_DATASET if example.name == "tree-automata-empty_test.v-dta_app_ne_inc_3")
TEST_EXAMPLE_PREFIX = COQGYM_TEST_SAMPLED_PERFECT_PREFIX[TEST_EXAMPLE.name]

print(TEST_EXAMPLE.name)
pprint(TEST_EXAMPLE)
print(TEST_EXAMPLE_PREFIX)

In [ ]:
coq = Coq(lemma_location=TEST_EXAMPLE.location)

In [ ]:
result = coq.run_code(TEST_EXAMPLE.proposition_command + "\nProof.\n" + TEST_EXAMPLE_PREFIX)
result

In [ ]:
pprint(result.fg_goals)

In [ ]:
def prefix_for_idx(idx: int):
    # repeat 'admit.' idx times
    return ' admit. ' * idx

[TEST_EXAMPLE_PREFIX + prefix_for_idx(i) for i in range(len(result.fg_goals))]

In [ ]:
def make_subgoal_examples(example: Example, perfect_prefix: str) -> t.List[Example]:
    if perfect_prefix.strip() == '':
        # if no perfect prefix, don't bother solving this example, it's covered in other test sets
        return []
    
    coq = Coq(lemma_location=example.location)
    print(example.name)
    result = coq.run_code(example.proposition_command + "\nProof.\n" + perfect_prefix)
    if not isinstance(result, c.contexts.ProofContext):
        raise ValueError("No context found in result.")
    
    return [Example(
        location=example.location,
        proposition_command=example.proposition_command,
        gold_standard_proof = example.gold_standard_proof,
        hint = example.hint,
        proof_prefix=perfect_prefix + prefix_for_idx(i) + ' - ',
        tag=f"subgoal{i}"
    ) for i in range(len(result.fg_goals))]

In [ ]:
SUBGOALS_GROUPED_DATASET = [
    make_subgoal_examples(example, COQGYM_TEST_SAMPLED_PERFECT_PREFIX[example.name])
    for example in COQGYM_TEST_SAMPLED_DATASET
    if example.name in COQGYM_TEST_SAMPLED_PERFECT_PREFIX
]
SUBGOAL_DATASET = [example 
                   for examples in SUBGOALS_GROUPED_DATASET
                   for example in examples ]

print(len(SUBGOAL_DATASET))

In [ ]:
for example in SUBGOAL_DATASET:
    print(example.name)

In [ ]:
# pickle the dev set to ./COQ_WIGDERSON_DEV_PERFECT_SUBGOALS.pkl
TEST_SAMPLED_FILE = current_directory / "../../data/COQGYM_TEST_PERFECT_SUBGOALS.pkl"
with open(TEST_SAMPLED_FILE, "wb") as f:
    pickle.dump(SUBGOAL_DATASET, f)